In [ ]:
import os
import scopesim
import synphot
import pytest
import scopesim
import numpy as np
import astropy
import scopesim_templates
import scopesim_targets
from astropy.io.fits import HDUList
from matplotlib import pyplot as plt
from matplotlib.colors import LogNorm
from scopesim import rc
from scopesim.source.source_templates import star_field
import scopesim_templates as sim_tp
from scopesim.optics.fov_manager import FOVManager
from astropy.io import fits
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
from scopesim_templates.stellar import star_field
from spextra import Spextrum
from scopesim_targets.point_source import Star
from scopesim_targets.extended_source import Sersic
from astropy import units as u

In [ ]:
cmds = scopesim.UserCommands(use_instrument="DREAMS")
dreams = scopesim.OpticalTrain(cmds)
cmds["!OBS.dit"] = 60 * 60
cmds["!DET.bin_size"] = 1
cmds["!OBS.sky.bg_mag"] = 14.9
cmds["!OBS.sky.filter_name"] = "J"
cmds["SIM.sub_pixel.flag"] = True
dreams = scopesim.OpticalTrain(cmds)
dreams["detector_linearity"].include = False

In [ ]:
source = star_field(n=15000,mmin=15, mmax=25, width=10000, height=10000)
supernova = Star(
    position=(1, 0),  # offset from center in arcsec
    brightness=("J", -15),  # point source brightness in J-band in mag
    spectrum=Spextrum("sne/sn1a").redshift(z=0.0025),  # SN1a spectrum
)
galaxy_1 = Sersic(
    params={"n": 4,
        "r_eff": 400*u.arcsec,
        "amplitude":10 ,
        "ellip": 0.625,
        "theta": 0.87,
        "c": -0.1
    },
    position=(0.0, 0.0),
    spectrum=Spextrum("brown/NGC0628").redshift(z=0.0025),
    brightness=("J", 8.48),
)

simulation_params = {
    "pixel_scale": 2.48 * u.arcsec/u.pix,  # pixel_scale off dreams
    "height": 500,
    "width": 500,
}

In [ ]:
source_1 = source + supernova.to_source() + galaxy_1.to_source(simulation_params)

source_1.fields[1].field["x"] = 2300 # supernova position
source_1.fields[1].field["y"] = 2500
source_1.fields[2].shift(2200, 2500) # Galaxy position

In [ ]:
dreams.observe(source_1)
hdul = dreams.readout('dreams_supernova.fits')
data = hdul[0][2].data  # Dreams detector 1

In [ ]:
plt.figure(figsize=(20,20))
plt.imshow(data, origin="lower", norm=LogNorm(vmin=9300000,vmax=9400000))
plt.colorbar()
plt.title("DREAMS Detector 1")
plt.xlabel("X Pixels")
plt.ylabel("Y Pixels")
plt.show()    